In [6]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/lunlun/Downloads/Github/textmining_grp6/webapp")

# Load sources
labeled     = pd.read_parquet(DATA_DIR / "labeled_sentiment.parquet")
sent_map    = pd.read_csv(DATA_DIR / "sent_chunk_map.csv")
topic_chunk = pd.read_parquet(DATA_DIR / "topic_chunk_df.parquet")
topic_dist  = pd.read_parquet(DATA_DIR / "topic_dist_df.parquet")

# ── 1. Attach chunk_id to every sentence (global sentence_id is the key)
df = labeled.merge(sent_map, on="sentence_id", how="left")

# ── 2. Attach topic_label + year via chunk_id
df = df.merge(
    topic_chunk[["chunk_id", "doc_id", "year", "topic_label"]],
    on=["chunk_id", "doc_id"],
    how="left",
)

# ── 3. Attach per-doc topic weight (doc_id + topic_label)
df = df.merge(
    topic_dist[["doc_id", "topic_label", "weight"]].rename(columns={"weight": "topic_weight"}),
    on=["doc_id", "topic_label"],
    how="left",
)

# ── 4. Neutral probability = 1 - pos - neg (clipped to [0,1])
df["neu"] = (1 - df["pos"].fillna(0) - df["neg"].fillna(0)).clip(0, 1)

# ── 5. Select & rename to final schema
FINAL_DF = df[[
    "doc_id", "company", "quarter", "year",
    "topic_label", "topic_weight",
    "sentence", "sentiment_score", "label",
    "pos", "neg", "neu",
    "sentence_id",
]].rename(columns={"sentiment_score": "score"})

print(FINAL_DF.shape)
print(FINAL_DF.dtypes)
FINAL_DF.head(5)


(452390, 13)
doc_id           object
company          object
quarter          object
year              int64
topic_label      object
topic_weight    float64
sentence         object
score           float32
label            object
pos             float32
neg             float32
neu             float32
sentence_id       int64
dtype: object


,doc_id,company,quarter,year,topic_label,topic_weight,sentence,score,label,pos,neg,neu,sentence_id
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,Preferred Stock & Equity Options,0.166667,presently the companys product line includes a...,0.000653,neutral,0.000687,0.000033,0.999280,0
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,Preferred Stock & Equity Options,0.166667,approximately NUM of all sales were generated ...,-0.000011,neutral,0.000033,0.000043,0.999924,1
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,Preferred Stock & Equity Options,0.166667,on an ongoing basis we re-evaluate our judgmen...,-0.000084,neutral,0.000019,0.000103,0.999878,2
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,Preferred Stock & Equity Options,0.166667,we base our estimates and judgments on our his...,-0.000004,neutral,0.000034,0.000037,0.999929,3
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,Preferred Stock & Equity Options,0.166667,actual results may differ from these estimates...,-0.000026,neutral,0.000023,0.000049,0.999928,4


In [7]:
# Sanity checks
null_counts = FINAL_DF.isnull().sum()
print("Null counts:\n", null_counts[null_counts > 0])

print(f"\nRows:      {len(FINAL_DF):,}")
print(f"Companies: {FINAL_DF.company.nunique()}")
print(f"Doc IDs:   {FINAL_DF.doc_id.nunique()}")
print(f"Topics:    {FINAL_DF.topic_label.nunique()}")
print(f"Labels:    {FINAL_DF.label.value_counts().to_dict()}")


Null counts:
 Series([], dtype: int64)

Rows:      452,390
Companies: 473
Doc IDs:   17560
Topics:    100
Labels:    {'neutral': 377900, 'positive': 38319, 'negative': 36171}


In [8]:
# Save
FINAL_DF.to_parquet("final_df.parquet", index=False)
print("Saved → final_df.parquet")


Saved → final_df.parquet
